# 并行机调度问题 (PMS)

**类别：** 调度

来源：[https://www.hexaly.com/templates/parallel-machine-scheduling-problem-pms](https://www.hexaly.com/templates/parallel-machine-scheduling-problem-pms)


## 问题描述

**并行机调度问题 (PMS)** 是调度文献中的经典问题。它由在一组并行机上调度任务组成。在最基本的表述中，所有机器都是相同的（即任务可以在任何机器上调度），并且每个任务只能由一台机器处理。

目标是找到一个使 makespan（所有任务的最大完成时间）最小的调度方案。这种并行机调度问题 (PMS) 的变体也被称为 **P || Cmax**，其中 P 表示存在相同的并行机，而 **Cmax** 表示优化准则是 makespan。

	

### 建模要点

- 添加 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 建模任务到资源的分配
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每台机器的 makespan


## 数据

为了说明此示例，我们从文献中选取了一组实例。并行机调度 (PMS) 实例的格式如下：

- 第一行：

- 任务数量；
- 机器数量；
- 第二行：与每个任务相关联的长度。


## 模型

并行机调度 (PMS) 的 Hexaly 模型使用 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。对每台机器，我们定义一个 set 变量表示分配给该机器的任务集合。我们将 set 变量约束为构成一个 partition，确保每个任务恰好在一台机器上调度。

我们使用对 set 的可变参数 **sum** 算子和一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算一台机器的 makespan，该函数返回与任何任务索引相关联的长度。请注意，求和中的项数在搜索过程中会动态变化，set 的大小也会变化。

我们使用 **max** 算子从每台机器各自的 makespan 中提取全局 makespan。该表达式定义了搜索过程中需要最小化的目标函数。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math

if len(sys.argv) < 2:
    print("Usage: python parallel_scheduling.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)

with hexaly.optimizer.HexalyOptimizer() as optimizer:
    # Read instance data
    filename = sys.argv[1]

    with open(filename) as f:
        line = f.readline()
        _, _, n_tasks, n_machines = [elem for elem in line.split()]

        # Number of tasks
        n_tasks = int(n_tasks)

        # Number of machines
        n_machines = int(n_machines)

        # Lengths of the tasks
        task_lengths = [int(elem) for elem in f.readline().split() if elem != '0']

        # Lowerbound on the makespan
        makespan_lb = int(sum(task_lengths) / n_machines) 

    # Declare the optimization model
    model = optimizer.model

    # Set decisions: machineTasks[k] represents the tasks assigned to machine k
    machine_tasks = [model.set(n_tasks) for _ in range(n_machines)]

    # Each task must be scheduled on exactly one machine
    model.constraint(model.partition(machine_tasks))

    # Create an array and a lambda function to retrieve the tasks' lengths
    lengths = model.array(task_lengths)
    lengths_lambda = model.lambda_function(lambda i: lengths[i])

    # Minimize the makespan
    machine_makespan = [model.sum(i, lengths_lambda) for i in machine_tasks]
    makespan = model.max(machine_makespan)
    model.minimize(makespan)

    model.close()

    # Parametrize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 5

    # Stop the search if the lower threshold is reached
    optimizer.param.set_objective_threshold(0, makespan_lb)

    optimizer.solve()

    # Write the solution in a file
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as f:
            for k in range(n_machines):
                f.write("Makespan machine {}: {} | Items: ".format(k, machine_makespan[k].value))
                if len(machine_tasks[k].value) == 0:
                    continue
                for e in machine_tasks[k].value:
                    f.write("%d " % e)
                f.write("\n")
